In [1]:
#@title 🧬 Peptide Property-Based Benchmarking { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header {
        background: linear-gradient(135deg, #ffd1dc, #ffc2d1, #ffe4ec);
        border-radius: 22px;
        padding: 28px 30px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 10px 25px rgba(255, 182, 193, 0.4);
        margin-bottom: 20px;
    }

    .title {
        text-align: center;
        color: #db2777;
        margin: 0 0 6px 0;
        font-size: 26px;
        font-weight: bold;
    }

    .toggle-btn {
        background: white;
        color: #db2777;
        border: 1px solid #fbcfe8;
        padding: 9px 16px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        margin: 8px 6px 0 0;
        transition: all 0.2s;
    }

    .toggle-btn:hover {
        background: #fce7f3;
        transform: translateY(-1px);
    }
</style>
<div class="main-header" id="headerBox">
    <div class="title">🧬 Peptide Property-Based Benchmarking</div>
    <div style="text-align:center; color:#4b5563; font-size:15px; margin-bottom:12px;">
        RL vs Pseudo-RL vs Hybrid and Advanced &nbsp;|&nbsp; vs Natural Reference Peptides
    </div>
    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #db2777, transparent); margin:16px 0;">
    <div style="font-size:14.8px; line-height:1.75; color:#374151;">
        This notebook benchmarks generated peptides (RL / Pseudo-RL / Hybrid and Advanced / ...) against
        <b>natural reference peptides</b>, on two complementary levels:

        <ul>
            <li><b>Literature-defined ranges</b> — fixed, absolute thresholds from published studies (e.g. AMP length 10–50 aa, net charge +2 to +9, pI 3.38–10.17).</li>
            <li><b>Statistical similarity to your uploaded Natural set</b> — Mann-Whitney U, Kolmogorov-Smirnov, Wasserstein distance, and empirical P5–P95 compliance.</li>
        </ul>
        Both are reported at the <b>Model</b> level and, separately, at the <b>Generation-Strategy</b> level
        (RL / Pseudo-RL / Hybrid and Advanced), since Strategy is a generation-method dimension independent of the biological Category (AMP/MHC/Cyclic).

        <br>
        <b>🟦 Blue/colored cards</b> = explanation.
        <b>⚙️ Gray code cells</b> = the code that runs it.
        Run cells top to bottom.
    </div>
</div>
<button class="toggle-btn" onclick="toggleHeader()">Hide Header</button>
<script>
function toggleHeader() {
    const box = document.getElementById("headerBox");
    const btns = document.querySelectorAll(".toggle-btn");
    const headerBtn = btns[0];

    if (box.style.display === "none") {
        box.style.display = "block";
        headerBtn.textContent = "Hide Header";
    } else {
        box.style.display = "none";
        headerBtn.textContent = "Show Header";
    }
}
</script>
"""))


In [2]:
#@title ⚙️ 1. Setup — Install & Import { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-setup {
        background: linear-gradient(135deg, #bfe6fb, #cdeffd, #e0f7ff);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(125, 211, 252, 0.4);
        margin-bottom: 16px;
    }
    .title-setup {
        text-align: center;
        color: #0284c7;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-setup {
        background: white;
        color: #0284c7;
        border: 1px solid #bae6fd;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-setup:hover {
        background: #e0f2fe;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-setup" id="headerBox_setup">
    <div class="title-setup">⚙️ 1. Setup — Install &amp; Import</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #0284c7, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        Installs BioPython / pandas / numpy / scipy / matplotlib / seaborn, and imports everything used below.
    </div>
</div>
<button class="toggle-btn-setup" onclick="toggleHeader_setup()">Hide card</button>
<script>
function toggleHeader_setup() {
    const box = document.getElementById("headerBox_setup");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


# Install required packages
!pip install biopython pandas numpy scipy matplotlib seaborn -q

from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import pandas as pd
import numpy as np
from scipy import stats
import re
import io
import base64
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 17.7 MB/s eta 0:00:00


In [3]:
#@title 📤 2. Upload FASTA Files { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-upload {
        background: linear-gradient(135deg, #ffd1dc, #ffc2d1, #ffe4ec);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(255, 182, 193, 0.4);
        margin-bottom: 16px;
    }
    .title-upload {
        text-align: center;
        color: #db2777;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-upload {
        background: white;
        color: #db2777;
        border: 1px solid #fbcfe8;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-upload:hover {
        background: #fce7f3;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-upload" id="headerBox_upload">
    <div class="title-upload">📤 2. Upload FASTA Files</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #db2777, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        Upload the generated-peptide FASTA file first, then the three natural reference sets (one per category).
        Expected header format for the <b>main</b> file: <code>&gt;ModelName_ID (Type)</code> —
        e.g. <code>&gt;HMAMP_001 (Antimicrobial)</code>. Natural files don't need model/type tags in the header;
        their category is fixed by which upload slot they go into.
    </div>
</div>
<button class="toggle-btn-upload" onclick="toggleHeader_upload()">Hide card</button>
<script>
function toggleHeader_upload() {
    const box = document.getElementById("headerBox_upload");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


print("Step 1/4 - Upload the FASTA file with your RL/model-generated peptides:")
uploaded_main = files.upload()
main_fasta_file = list(uploaded_main.keys())[0]

print("\nStep 2/4 - Upload the FASTA file with natural AMP peptides:")
uploaded_amp = files.upload()
amp_fasta_file = list(uploaded_amp.keys())[0]

print("\nStep 3/4 - Upload the FASTA file with natural MHC-binder peptides:")
uploaded_mhc = files.upload()
mhc_fasta_file = list(uploaded_mhc.keys())[0]

print("\nStep 4/4 - Upload the FASTA file with natural cyclic peptides:")
uploaded_cyclic = files.upload()
cyclic_fasta_file = list(uploaded_cyclic.keys())[0]

NATURAL_FILES = {
    "AMP": amp_fasta_file,
    "MHC": mhc_fasta_file,
    "Cyclic": cyclic_fasta_file,
}


Step 1/4 - Upload the FASTA file with your RL/model-generated peptides:


Saving Property-based benchmarking.fasta to Property-based benchmarking.fasta

Step 2/4 - Upload the FASTA file with natural AMP peptides:


Saving AMP-natural-aps.unmc.edu -100.fasta to AMP-natural-aps.unmc.edu -100.fasta

Step 3/4 - Upload the FASTA file with natural MHC-binder peptides:


Saving reference_mhc_100.fasta to reference_mhc_100.fasta

Step 4/4 - Upload the FASTA file with natural cyclic peptides:


Saving reference_cyclic_100_standard.fasta to reference_cyclic_100_standard.fasta


In [4]:
#@title 🧹 3. Data Preparation Helpers { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-dataprep {
        background: linear-gradient(135deg, #bfe6fb, #cdeffd, #e0f7ff);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(125, 211, 252, 0.4);
        margin-bottom: 16px;
    }
    .title-dataprep {
        text-align: center;
        color: #0284c7;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-dataprep {
        background: white;
        color: #0284c7;
        border: 1px solid #bae6fd;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-dataprep:hover {
        background: #e0f2fe;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-dataprep" id="headerBox_dataprep">
    <div class="title-dataprep">🧹 3. Data Preparation Helpers</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #0284c7, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        This single cell defines every helper used to turn a raw FASTA record into a clean, labeled row:
        <ul>
        <li><b>validate_sequence</b> — removes whitespace, validates against the 20 standard amino acids, and excludes the entire peptide if any non-canonical residue is present.</li>
        <li><b>extract_info / normalize_type</b> — pulls the model name and biological <code>Category</code>
        (AMP / MHC / Cyclic) from the header.</li>
        <li><b>classify_strategy</b> (NEW) — a <i>separate</i> generation-method label
        (<code>RL</code> / <code>Pseudo-RL</code> / <code>Hybrid and Advanced</code> / <code>Other</code>), independent
        of Category, since one strategy can produce peptides across multiple categories. Fill in
        <code>STRATEGY_OVERRIDE</code> below if your naming convention doesn't contain these keywords.</li>
        <li><b>compute_properties</b> — Length, MW, GRAVY, sequence-based estimated pI, NetCharge (charge at pH 7, Biopython approximation),
        Aliphatic Index.</li>
        </ul>
    </div>
</div>
<button class="toggle-btn-dataprep" onclick="toggleHeader_dataprep()">Hide card</button>
<script>
function toggleHeader_dataprep() {
    const box = document.getElementById("headerBox_dataprep");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

def validate_sequence(seq):
    """
    Remove whitespace only. If ANY non-canonical amino-acid symbol remains,
    exclude the ENTIRE peptide from quantitative benchmarking.
    """
    seq = re.sub(r"\\s+", "", str(seq).upper())
    invalid = sorted(set(seq) - STANDARD_AA)
    if invalid:
        return None, invalid
    return seq, []

def standardize_header(header):
    header = header.strip()
    header = re.sub(r"\s+", " ", header)
    return header

def extract_info(header):
    header = standardize_header(header)
    type_match = re.search(r"\((.*?)\)", header)
    pep_type = type_match.group(1).strip() if type_match else "Unknown"
    model_match = re.match(r"^>?\s*([A-Za-z][A-Za-z0-9]*)", header)
    model_name = model_match.group(1).strip().upper() if model_match else "UNKNOWN"
    return model_name, pep_type

def normalize_type(pep_type):
    t = pep_type.lower()
    if "antimicrobial" in t or "amp" in t:
        return "AMP"
    elif "mhc" in t or "immunogen" in t or "binding" in t:
        return "MHC"
    elif "cyclic" in t or "macrocyclic" in t:
        return "Cyclic"
    else:
        return "Mixed"

# --- Generation strategy (NEW): independent of Category ---
# Explicit overrides checked first, e.g. STRATEGY_OVERRIDE = {"HMAMP": "RL", "GANHYB": "Hybrid and Advanced"}
STRATEGY_OVERRIDE = {
    # Explicit RL / policy-gradient / PPO / REINFORCE-type approaches
    "HMAMP": "RL",
    "TARSA": "RL",
    "AMP": "RL",          # AMP-RL
    "RR": "RL",           # RR-ADS
    "PEPGAN": "RL",
    "RLPMIEC": "RL",

    # Hybrid / advanced RL frameworks
    "DIFF": "Hybrid and Advanced",        # Diff-AMP: diffusion + RL
    "AMPAINTER": "Hybrid and Advanced",   # RL policy + language model
    "ULTRAMUTATE": "Hybrid and Advanced", # RL + MCTS
    "CYC": "Hybrid and Advanced",         # CYC_BUILDER: MCTS + RL
    "HIGHPLAY": "Hybrid and Advanced",    # MCTS + policy-value network

    # RL-inspired / pseudo-RL approaches
    "OH": "Pseudo-RL",     # Oh et al.
    "VAE": "Pseudo-RL",    # VAE-MH
    "RF": "Pseudo-RL",     # RF-AF3
}

STRATEGY_PATTERNS = [
    ("Pseudo-RL", re.compile(r"pseudo[\s\-_]?rl", re.IGNORECASE)),
    ("Hybrid and Advanced", re.compile(r"hybrid|advanced", re.IGNORECASE)),
    ("RL", re.compile(r"\brl\b|reinforcement", re.IGNORECASE)),
]

def classify_strategy(model_name, raw_header):
    if model_name in STRATEGY_OVERRIDE:
        return STRATEGY_OVERRIDE[model_name]
    text = f"{model_name} {raw_header}"
    for label, pattern in STRATEGY_PATTERNS:
        if pattern.search(text):
            return label
    return "Other"

# --- Properties ---
PROPERTIES = ["Length", "MW", "GRAVY", "pI", "NetCharge", "AliphaticIndex"]

def isoelectric_point_extended(analysis, min_pH=0.0, max_pH=14.0, tol=1e-6, max_iter=200):
    """
    Calculate sequence-based estimated pI using Biopython's charge_at_pH model while searching the
    full pH 0-14 interval. This avoids the ~4.05 lower-bound artifact of
    ProteinAnalysis.isoelectric_point() for very acidic peptides.
    """
    low, high = float(min_pH), float(max_pH)
    q_low = analysis.charge_at_pH(low)
    q_high = analysis.charge_at_pH(high)

    if np.isclose(q_low, 0.0, atol=1e-12):
        return low
    if np.isclose(q_high, 0.0, atol=1e-12):
        return high
    if q_low * q_high > 0:
        return np.nan

    for _ in range(max_iter):
        mid = (low + high) / 2.0
        q_mid = analysis.charge_at_pH(mid)
        if abs(q_mid) < 1e-10 or (high - low) < tol:
            return mid
        if q_mid > 0:
            low = mid
        else:
            high = mid
    return (low + high) / 2.0

def compute_properties(seq):
    if len(seq) == 0:
        return None
    analysis = ProteinAnalysis(seq)
    return {
        "Length": len(seq),
        "MW": analysis.molecular_weight(),
        "GRAVY": analysis.gravy(),
        "pI": isoelectric_point_extended(analysis),
        "NetCharge": analysis.charge_at_pH(7.0),
        "AliphaticIndex": (
            (seq.count("A") +
             2.9 * seq.count("V") +
             3.9 * seq.count("I") +
             3.9 * seq.count("L")) * 100 / len(seq)
        )
    }


In [5]:
#@title 📄 4. Parse FASTA Files → Build Dataset { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-parse {
        background: linear-gradient(135deg, #ffd1dc, #ffc2d1, #ffe4ec);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(255, 182, 193, 0.4);
        margin-bottom: 16px;
    }
    .title-parse {
        text-align: center;
        color: #db2777;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-parse {
        background: white;
        color: #db2777;
        border: 1px solid #fbcfe8;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-parse:hover {
        background: #fce7f3;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-parse" id="headerBox_parse">
    <div class="title-parse">📄 4. Parse FASTA Files → Build Dataset</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #db2777, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        Parses the main (generated) file and all three natural files into one combined DataFrame <code>df</code>,
        tagging every row with <code>Model</code>, <code>Category</code>, and (for generated peptides)
        <code>Strategy</code>. Also prints diagnostics: unparsed headers, models that fell back to
        <code>Other</code> strategy, and (Category, Model) groups too small (&lt;3) for statistical testing later.
    </div>
</div>
<button class="toggle-btn-parse" onclick="toggleHeader_parse()">Hide card</button>
<script>
function toggleHeader_parse() {
    const box = document.getElementById("headerBox_parse");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


rows = []
excluded_rows = []

# Generated/model peptides
for record in SeqIO.parse(main_fasta_file, "fasta"):
    raw_seq = str(record.seq)
    model_name, pep_type = extract_info(record.description)
    category = normalize_type(pep_type)
    strategy = classify_strategy(model_name, record.description)

    valid_seq, invalid = validate_sequence(raw_seq)

    # IMPORTANT: exclude the whole peptide if any invalid/non-canonical residue exists
    if valid_seq is None:
        excluded_rows.append({
            "ID": record.id,
            "Source": "Generated",
            "Model": model_name,
            "Category": category,
            "RawSequence": re.sub(r"\\s+", "", raw_seq.upper()),
            "InvalidAA": ",".join(invalid),
            "Reason": "Excluded: contains non-canonical amino-acid symbol(s)"
        })
        continue

    props = compute_properties(valid_seq)
    if props is None:
        excluded_rows.append({
            "ID": record.id,
            "Source": "Generated",
            "Model": model_name,
            "Category": category,
            "RawSequence": re.sub(r"\\s+", "", raw_seq.upper()),
            "InvalidAA": None,
            "Reason": "Excluded: empty/invalid sequence"
        })
        continue

    rows.append({
        "ID": record.id,
        "Model": model_name,
        "Strategy": strategy,
        "RawType": pep_type,
        "Category": category,
        "CleanSeq": valid_seq,
        "InvalidAA": None,
        **props
    })

# Natural reference peptides
for category, fasta_path in NATURAL_FILES.items():
    for record in SeqIO.parse(fasta_path, "fasta"):
        raw_seq = str(record.seq)
        valid_seq, invalid = validate_sequence(raw_seq)

        if valid_seq is None:
            excluded_rows.append({
                "ID": record.id,
                "Source": f"Natural {category}",
                "Model": "Natural",
                "Category": category,
                "RawSequence": re.sub(r"\\s+", "", raw_seq.upper()),
                "InvalidAA": ",".join(invalid),
                "Reason": "Excluded: contains non-canonical amino-acid symbol(s)"
            })
            continue

        props = compute_properties(valid_seq)
        if props is None:
            excluded_rows.append({
                "ID": record.id,
                "Source": f"Natural {category}",
                "Model": "Natural",
                "Category": category,
                "RawSequence": re.sub(r"\\s+", "", raw_seq.upper()),
                "InvalidAA": None,
                "Reason": "Excluded: empty/invalid sequence"
            })
            continue

        rows.append({
            "ID": record.id,
            "Model": "Natural",
            "Strategy": "Natural",
            "RawType": f"Natural {category}",
            "Category": category,
            "CleanSeq": valid_seq,
            "InvalidAA": None,
            **props
        })

df = pd.DataFrame(rows)
excluded_df = pd.DataFrame(excluded_rows)

print(f"Total INCLUDED sequences: {len(df)}")
print(f"Total EXCLUDED sequences: {len(excluded_df)}")

if len(excluded_df) > 0:
    print("\nExcluded sequences:")
    display(excluded_df)
    excluded_df.to_csv("excluded_sequences.csv", index=False)
    print("\nSaved exclusion log: excluded_sequences.csv")
print(f"Total sequences: {len(df)} | Models: {list(df['Model'].unique())} | "
      f"Strategies: {list(df['Strategy'].unique())} | Categories: {list(df['Category'].unique())}")

unresolved = df[(df["Model"] == "UNKNOWN") | (df["RawType"] == "Unknown")]
unresolved = unresolved[unresolved["Model"] != "Natural"]
if len(unresolved) > 0:
    print(f"\nWARNING: {len(unresolved)} sequence(s) could not be fully parsed. "
          f"Expected header format: '>ModelName_ID (Type)'. Examples:")
    print(unresolved[["ID", "Model", "RawType", "Category"]].head(10).to_string(index=False))

other_models = sorted(df[(df["Strategy"] == "Other")]["Model"].unique())
if other_models:
    print(f"\nNOTE: models labeled 'Other' strategy (not matched by keyword/override): {other_models}. "
          f"Add them to STRATEGY_OVERRIDE above if needed.")

small_models = df[df["Model"] != "Natural"].groupby(["Category", "Model"]).size()
small_models = small_models[small_models < 3]
if len(small_models) > 0:
    print(f"\nWARNING: these (Category, Model) groups have < 3 sequences and will be skipped "
          f"in the statistical comparison:")
    print(small_models.to_string())


Total INCLUDED sequences: 443
Total EXCLUDED sequences: 1

Excluded sequences:


,ID,Source,Model,Category,RawSequence,InvalidAA,Reason
0,HMAMP_4,Generated,HMAMP,AMP,ARRYKFFIRLARVLKNYGRVJTGLETYRDLIGP,J,Excluded: contains non-canonical amino-acid sy...



Saved exclusion log: excluded_sequences.csv
Total sequences: 443 | Models: ['HMAMP', 'DIFF', 'TARSA', 'ULTRAMUTATE', 'OH', 'AMP', 'AMPAINTER', 'RR', 'PEPGAN', 'RLPMIEC', 'CYC', 'HIGHPLAY', 'VAE', 'RF', 'Natural'] | Strategies: ['RL', 'Hybrid and Advanced', 'Pseudo-RL', 'Natural'] | Categories: ['AMP', 'Mixed', 'MHC', 'Cyclic']

Category  Model      
MHC       RLPMIEC        2
          ULTRAMUTATE    2
Mixed     RF             1


In [6]:
#@title 🔎 4B. pI Quality Control — Publication Summary + Supplementary Audit { display-mode: "form" }
# Primary analysis uses ONLY the corrected pI stored in df["pI"].
# The original/default Biopython pI is recalculated only for QC/provenance.
# The full before-vs-after audit is exported to CSV and is not shown as a large table
# in the main publication HTML.

pi_audit_rows = []
for _, row in df.iterrows():
    analysis = ProteinAnalysis(row["CleanSeq"])
    original_pi = analysis.isoelectric_point()
    corrected_pi = row["pI"]
    pi_audit_rows.append({
        "ID": row["ID"],
        "Category": row["Category"],
        "Model": row["Model"],
        "Sequence": row["CleanSeq"],
        "pI_original_Biopython": original_pi,
        "pI_corrected": corrected_pi,
        "Difference_corrected_minus_original": corrected_pi - original_pi,
        "Charge_at_pH_4.05": analysis.charge_at_pH(4.05),
        "Original_at_4.050028": bool(np.isclose(original_pi, 4.050028, atol=1e-6)),
        "Corrected_at_4.050028": bool(np.isclose(corrected_pi, 4.050028, atol=1e-6)),
    })

pi_audit_df = pd.DataFrame(pi_audit_rows)
cyclic_nat_audit = pi_audit_df[
    (pi_audit_df["Category"] == "Cyclic") &
    (pi_audit_df["Model"] == "Natural")
].copy()
affected_cyclic = cyclic_nat_audit[cyclic_nat_audit["Original_at_4.050028"]].copy()

def _safe_sd(series):
    return float(series.std(ddof=1)) if len(series) > 1 else np.nan

pi_qc_summary = pd.DataFrame([
    {"QC_metric": "Natural Cyclic sample size", "Value": int(len(cyclic_nat_audit))},
    {"QC_metric": "Original Biopython pI ≈ 4.050028, n (%)",
     "Value": f"{int(cyclic_nat_audit['Original_at_4.050028'].sum())} ({100*cyclic_nat_audit['Original_at_4.050028'].mean():.1f}%)"},
    {"QC_metric": "Corrected pI ≈ 4.050028, n (%)",
     "Value": f"{int(cyclic_nat_audit['Corrected_at_4.050028'].sum())} ({100*cyclic_nat_audit['Corrected_at_4.050028'].mean():.1f}%)"},
    {"QC_metric": "Affected peptides negative at pH 4.05, n/N",
     "Value": (f"{int((affected_cyclic['Charge_at_pH_4.05'] < 0).sum())}/{len(affected_cyclic)}" if len(affected_cyclic) else "0/0")},
    {"QC_metric": "Original Natural Cyclic pI mean ± SD",
     "Value": f"{cyclic_nat_audit['pI_original_Biopython'].mean():.3f} ± {_safe_sd(cyclic_nat_audit['pI_original_Biopython']):.3f}"},
    {"QC_metric": "Corrected Natural Cyclic pI mean ± SD",
     "Value": f"{cyclic_nat_audit['pI_corrected'].mean():.3f} ± {_safe_sd(cyclic_nat_audit['pI_corrected']):.3f}"},
    {"QC_metric": "Corrected Natural Cyclic pI P5–P95",
     "Value": f"{cyclic_nat_audit['pI_corrected'].quantile(0.05):.3f}–{cyclic_nat_audit['pI_corrected'].quantile(0.95):.3f}"},
])

print("=" * 88)
print("pI QUALITY CONTROL — PUBLICATION SUMMARY")
print("=" * 88)
display(pi_qc_summary)

if len(affected_cyclic):
    n_negative = int((affected_cyclic["Charge_at_pH_4.05"] < 0).sum())
    print(f"\nAffected Natural Cyclic peptides negative at pH 4.05: {n_negative}/{len(affected_cyclic)}")

pi_audit_df.to_csv("Supplementary_pI_audit_before_after.csv", index=False)
df.to_csv("all_peptides_with_corrected_pI.csv", index=False)
pi_qc_summary.to_csv("pI_QC_publication_summary.csv", index=False)
print("\nSaved supplementary/provenance files:")
print("  - Supplementary_pI_audit_before_after.csv")
print("  - all_peptides_with_corrected_pI.csv")
print("  - pI_QC_publication_summary.csv")
print("\nAll downstream benchmarking uses df['pI'] = corrected pI only.")


pI QUALITY CONTROL — PUBLICATION SUMMARY


,QC_metric,Value
0,Natural Cyclic sample size,100
1,"Original Biopython pI ≈ 4.050028, n (%)",94 (94.0%)
2,"Corrected pI ≈ 4.050028, n (%)",0 (0.0%)
3,"Affected peptides negative at pH 4.05, n/N",94/94
4,Original Natural Cyclic pI mean ± SD,4.060 ± 0.050
5,Corrected Natural Cyclic pI mean ± SD,3.606 ± 0.277
6,Corrected Natural Cyclic pI P5–P95,3.240–4.062



Affected Natural Cyclic peptides negative at pH 4.05: 94/94

Saved supplementary/provenance files:
  - Supplementary_pI_audit_before_after.csv
  - all_peptides_with_corrected_pI.csv
  - pI_QC_publication_summary.csv

All downstream benchmarking uses df['pI'] = corrected pI only.


In [7]:
#@title 📊 5. Benchmark Baselines { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-baseline {
        background: linear-gradient(135deg, #bfe6fb, #cdeffd, #e0f7ff);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(125, 211, 252, 0.4);
        margin-bottom: 16px;
    }
    .title-baseline {
        text-align: center;
        color: #0284c7;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-baseline {
        background: white;
        color: #0284c7;
        border: 1px solid #bae6fd;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-baseline:hover {
        background: #e0f2fe;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-baseline" id="headerBox_baseline">
    <div class="title-baseline">📊 5. Benchmark Baselines — Natural Distribution &amp; Literature Ranges</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #0284c7, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        Two reference baselines, computed together here:
        <ul>
        <li><b>Natural reference distribution</b> per Category × Property (mean, SD, P5/P25/Median/P75/P95, Shapiro-Wilk
        normality check) — used later for statistical comparison.</li>
        <li><b>Literature Compliance Table</b> — fixed, absolute thresholds from published studies, independent of the specific
        Natural sample uploaded):</li>
        </ul>
        <table style="border-collapse:collapse;width:100%;font-size:13px;">
        <tr style="background:#0284c7;color:white;"><th style="padding:6px;">Property</th><th style="padding:6px;">Category</th><th style="padding:6px;">Range</th><th style="padding:6px;">Source</th></tr>
        <tr style="background:#e0f7ff;"><td style="padding:6px;text-align:center;">Length</td><td style="padding:6px;text-align:center;">AMP</td><td style="padding:6px;text-align:center;">10&ndash;50 aa</td><td style="padding:6px;text-align:center;">PMC3910807</td></tr>
        <tr><td style="padding:6px;text-align:center;">Length</td><td style="padding:6px;text-align:center;">MHC</td><td style="padding:6px;text-align:center;">typically 9-mer (8&ndash;10 ≈ 78%)</td><td style="padding:6px;text-align:center;">PMC6224037</td></tr>
        <tr style="background:#e0f7ff;"><td style="padding:6px;text-align:center;">Length</td><td style="padding:6px;text-align:center;">Cyclic</td><td style="padding:6px;text-align:center;">7&ndash;40 aa</td><td style="padding:6px;text-align:center;">PLOS Comp Biol 10.1371/journal.pcbi.1012290</td></tr>
        <tr><td style="padding:6px;text-align:center;">Net Charge</td><td style="padding:6px;text-align:center;">AMP</td><td style="padding:6px;text-align:center;">+2 to +9</td><td style="padding:6px;text-align:center;">Springer 10.1186/s40779-021-00343-2</td></tr>
        <tr style="background:#e0f7ff;"><td style="padding:6px;text-align:center;">pI</td><td style="padding:6px;text-align:center;">all</td><td style="padding:6px;text-align:center;">3.38&ndash;10.17</td><td style="padding:6px;text-align:center;">PMID 10726766</td></tr>
        </table>
        <br>GRAVY, Molecular Weight, and Aliphatic Index have no universal fixed range (task-dependent) — reported as
        descriptive statistics only.
    </div>
</div>
<button class="toggle-btn-baseline" onclick="toggleHeader_baseline()">Hide card</button>
<script>
function toggleHeader_baseline() {
    const box = document.getElementById("headerBox_baseline");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


natural_ref_rows = []
for category in sorted(df["Category"].unique()):
    nat_vals_all = df[(df["Category"] == category) & (df["Model"] == "Natural")]
    for prop in PROPERTIES:
        vals = nat_vals_all[prop].dropna().values
        n = len(vals)
        if n == 0:
            continue
        p5, p25, p50, p75, p95 = np.percentile(vals, [5, 25, 50, 75, 95]) if n >= 2 else (np.nan,) * 5
        if n >= 3:
            shapiro_stat, shapiro_p = stats.shapiro(vals)
            is_normal = shapiro_p > 0.05
        else:
            shapiro_stat, shapiro_p, is_normal = np.nan, np.nan, np.nan
        natural_ref_rows.append({
            "Category": category, "Property": prop, "N_Natural": n,
            "Mean": round(np.mean(vals), 3), "SD": round(np.std(vals), 3),
            "P5": round(p5, 3) if n >= 2 else np.nan, "P25": round(p25, 3) if n >= 2 else np.nan,
            "Median": round(p50, 3) if n >= 2 else np.nan, "P75": round(p75, 3) if n >= 2 else np.nan,
            "P95": round(p95, 3) if n >= 2 else np.nan,
            "Shapiro_p": round(shapiro_p, 4) if n >= 3 else np.nan, "Normal_Distribution": is_normal,
        })
natural_ref_df = pd.DataFrame(natural_ref_rows)

LITERATURE_RANGES = {
    "AMP":    {"Length": (10, 50), "NetCharge": (2, 9), "pI": (3.38, 10.17)},
    "MHC":    {"Length": (8, 10),                      "pI": (3.38, 10.17)},
    "Cyclic": {"Length": (7, 40),                      "pI": (3.38, 10.17)},
    "Mixed":  {                                        "pI": (3.38, 10.17)},
}

def literature_compliance_table(df):
    rows = []
    for (category, model), grp in df.groupby(["Category", "Model"]):
        ranges = LITERATURE_RANGES.get(category, {})
        row = {"Category": category, "Model": model, "N": len(grp)}
        for prop in ["Length", "NetCharge", "pI"]:
            row[f"Avg_{prop}"] = round(grp[prop].mean(), 3)
            if prop in ranges:
                lo, hi = ranges[prop]
                row[f"%_{prop}_in_[{lo}-{hi}]"] = round(np.mean((grp[prop] >= lo) & (grp[prop] <= hi)) * 100, 1)
            else:
                row[f"%_{prop}_in_range"] = np.nan
        row["Avg_GRAVY"] = round(grp["GRAVY"].mean(), 3)
        row["Avg_MW"] = round(grp["MW"].mean(), 1)
        row["Avg_AliphaticIndex"] = round(grp["AliphaticIndex"].mean(), 2)
        if category == "MHC":
            row["%_exactly_9mer"] = round(np.mean(grp["Length"] == 9) * 100, 1)
        rows.append(row)
    out = pd.DataFrame(rows)
    out["_natural_first"] = out["Model"] != "Natural"
    return out.sort_values(["Category", "_natural_first", "Model"]).drop(columns="_natural_first")

literature_table = literature_compliance_table(df)
print("Natural reference distribution:"); display(natural_ref_df)
print("\nLiterature Compliance Table:"); display(literature_table)


Property,Category,Range,Source
Length,AMP,10–50 aa,PMC3910807
Length,MHC,typically 9-mer (8–10 ≈ 78%),PMC6224037
Length,Cyclic,7–40 aa,PLOS Comp Biol 10.1371/journal.pcbi.1012290
Net Charge,AMP,+2 to +9,Springer 10.1186/s40779-021-00343-2
pI,all,3.38–10.17,PMID 10726766


Natural reference distribution:


,Category,Property,N_Natural,Mean,SD,P5,P25,Median,P75,P95,Shapiro_p,Normal_Distribution
0,AMP,Length,100,27.890,13.416,13.000,16.000,27.000,37.000,46.000,0.0000,False
1,AMP,MW,100,2999.506,1504.138,1369.733,1647.719,2664.213,3848.332,5139.322,0.0000,False
2,AMP,GRAVY,100,0.482,0.833,-0.886,-0.067,0.393,1.191,1.808,0.0488,False
3,AMP,pI,100,9.441,1.422,6.698,8.750,9.658,10.003,11.711,0.0000,False
4,AMP,NetCharge,100,3.116,2.774,-0.178,0.799,2.749,3.837,8.759,0.0000,False
5,AMP,AliphaticIndex,100,116.424,52.748,44.487,72.375,112.198,157.800,210.643,0.0086,False
6,Cyclic,Length,100,19.750,6.866,8.000,14.000,20.000,26.000,30.000,0.0006,False
7,Cyclic,MW,100,2322.297,774.964,961.109,1726.479,2262.659,2971.955,3507.121,0.0035,False
8,Cyclic,GRAVY,100,0.242,0.474,-0.517,-0.112,0.264,0.489,0.992,0.5425,True
9,Cyclic,pI,100,3.606,0.275,3.240,3.385,3.564,3.776,4.062,0.0001,False



Literature Compliance Table:


,Category,Model,N,Avg_Length,%_Length_in_[10-50],Avg_NetCharge,%_NetCharge_in_[2-9],Avg_pI,%_pI_in_[3.38-10.17],Avg_GRAVY,Avg_MW,Avg_AliphaticIndex,%_Length_in_[7-40],%_NetCharge_in_range,%_Length_in_[8-10],%_exactly_9mer,%_Length_in_range
4,AMP,Natural,100,27.890,97.0,3.116,53.0,9.441,79.0,0.482,2999.5,116.42,NaN,NaN,NaN,NaN,NaN
0,AMP,AMP,6,19.333,100.0,7.774,66.7,11.412,0.0,-1.277,2515.9,71.98,NaN,NaN,NaN,NaN,NaN
1,AMP,AMPAINTER,30,21.733,100.0,8.024,70.0,11.797,0.0,-0.727,2846.7,94.21,NaN,NaN,NaN,NaN,NaN
2,AMP,DIFF,6,14.500,66.7,1.442,50.0,8.047,50.0,0.519,1528.5,85.33,NaN,NaN,NaN,NaN,NaN
3,AMP,HMAMP,9,17.444,77.8,2.861,77.8,10.258,55.6,0.099,2071.7,113.19,NaN,NaN,NaN,NaN,NaN
5,AMP,PEPGAN,6,19.833,100.0,5.420,100.0,10.867,0.0,0.529,2215.1,145.29,NaN,NaN,NaN,NaN,NaN
6,AMP,RR,15,16.333,100.0,5.175,100.0,11.673,0.0,-0.425,2149.6,113.74,NaN,NaN,NaN,NaN,NaN
9,Cyclic,Natural,100,19.750,NaN,-5.171,NaN,3.606,78.0,0.242,2322.3,102.48,100.0,NaN,NaN,NaN,NaN
7,Cyclic,CYC,9,11.222,NaN,-1.946,NaN,4.373,100.0,-0.698,1337.1,79.47,100.0,NaN,NaN,NaN,NaN
8,Cyclic,HIGHPLAY,29,16.103,NaN,-1.125,NaN,5.670,100.0,-0.502,1867.9,50.18,100.0,NaN,NaN,NaN,NaN


In [8]:
#@title 📈 6. Model-Level Comparison & Ranking { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-modelcmp {
        background: linear-gradient(135deg, #ffd1dc, #ffc2d1, #ffe4ec);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(255, 182, 193, 0.4);
        margin-bottom: 16px;
    }
    .title-modelcmp {
        text-align: center;
        color: #db2777;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-modelcmp {
        background: white;
        color: #db2777;
        border: 1px solid #fbcfe8;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-modelcmp:hover {
        background: #fce7f3;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-modelcmp" id="headerBox_modelcmp">
    <div class="title-modelcmp">📈 6. Model-Level Comparison &amp; Ranking vs. Natural</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #db2777, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        For each Model, computes Mann-Whitney U, Kolmogorov-Smirnov, normalized Wasserstein distance, and empirical
        P5&ndash;P95 compliance against the Natural reference, then combines these into a composite ranking (Natural itself
        is shown as the 100%-compliance baseline, never ranked as a competitor).
    </div>
</div>
<button class="toggle-btn-modelcmp" onclick="toggleHeader_modelcmp()">Hide card</button>
<script>
function toggleHeader_modelcmp() {
    const box = document.getElementById("headerBox_modelcmp");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


comparison_rows = []
for category in sorted(df["Category"].unique()):
    nat_sub = df[(df["Category"] == category) & (df["Model"] == "Natural")]
    models = [m for m in df[df["Category"] == category]["Model"].unique() if m != "Natural"]
    for prop in PROPERTIES:
        nat_vals = nat_sub[prop].dropna().values
        if len(nat_vals) < 3:
            continue
        p5, p95 = np.percentile(nat_vals, [5, 95])
        nat_std = np.std(nat_vals) if np.std(nat_vals) > 0 else np.nan
        for model in models:
            model_vals = df[(df["Category"] == category) & (df["Model"] == model)][prop].dropna().values
            if len(model_vals) < 3:
                continue
            mw_u, mw_p = stats.mannwhitneyu(model_vals, nat_vals, alternative="two-sided")
            ks_stat, ks_p = stats.ks_2samp(model_vals, nat_vals)
            wass = stats.wasserstein_distance(model_vals, nat_vals)
            wass_norm = wass / nat_std if nat_std and not np.isnan(nat_std) else np.nan
            compliance = float(np.mean((model_vals >= p5) & (model_vals <= p95)) * 100)
            comparison_rows.append({
                "Category": category, "Model": model, "Property": prop,
                "N_Model": len(model_vals), "N_Natural": len(nat_vals),
                "MannWhitney_p": round(mw_p, 4), "Significant_Diff (p<0.05)": mw_p < 0.05,
                "KS_stat": round(ks_stat, 3), "KS_p": round(ks_p, 4),
                "Wasserstein_norm": round(wass_norm, 3) if not np.isnan(wass_norm) else np.nan,
                "Compliance_%_in_Natural_P5-P95": round(compliance, 1),
            })
comparison_df = pd.DataFrame(comparison_rows)

composite_rows = []
for (category, model), grp in comparison_df.groupby(["Category", "Model"]):
    composite_rows.append({
        "Category": category, "Model": model,
        "Composite_Compliance_%": round(grp["Compliance_%_in_Natural_P5-P95"].mean(), 1),
        "Composite_Distance": round(grp["Wasserstein_norm"].mean(skipna=True), 3),
        "Properties_Significantly_Different": int(grp["Significant_Diff (p<0.05)"].sum()),
        "Properties_Tested": len(grp),
    })
for category in sorted(df["Category"].unique()):
    composite_rows.append({
        "Category": category, "Model": "Natural", "Composite_Compliance_%": 100.0,
        "Composite_Distance": 0.0, "Properties_Significantly_Different": 0, "Properties_Tested": len(PROPERTIES),
    })
composite_df = pd.DataFrame(composite_rows)

rank_rows = []
for category, grp in composite_df.groupby("Category"):
    grp = grp.copy()
    rl_mask = grp["Model"] != "Natural"
    grp.loc[rl_mask, "Rank_by_Compliance"] = grp.loc[rl_mask, "Composite_Compliance_%"].rank(ascending=False, method="min").astype(int)
    grp.loc[rl_mask, "Rank_by_Distance"] = grp.loc[rl_mask, "Composite_Distance"].rank(ascending=True, method="min").astype(int)
    rank_rows.append(grp)
rank_df = pd.concat(rank_rows).sort_values(["Category", "Rank_by_Compliance"])

rl_rank_df = rank_df[rank_df["Model"] != "Natural"].copy()
rl_rank_df["Rank_by_Compliance"] = rl_rank_df["Rank_by_Compliance"].astype(int)
rl_rank_df["Rank_by_Distance"] = rl_rank_df["Rank_by_Distance"].astype(int)

best_per_category = rl_rank_df[rl_rank_df["Rank_by_Compliance"] == 1][
    ["Category", "Model", "Composite_Compliance_%", "Composite_Distance"]
]
print("Best model per category:"); display(best_per_category)


Best model per category:


,Category,Model,Composite_Compliance_%,Composite_Distance
4,AMP,PEPGAN,97.2,0.840
6,Cyclic,CYC,63.0,1.633


In [9]:
#@title 🧪 6B. Feature Contribution & Leave-One-Feature-Out Sensitivity { display-mode: "form" }
# Publication sensitivity analysis. No new feature weights are introduced here. All six features retain equal weight.
# The analysis quantifies effective contribution and temporary feature removal only.

# ---------- A) Percentage contribution to Composite_Distance ----------
# Since Composite_Distance is the arithmetic mean of six normalized Wasserstein
# distances, each property's effective share is its distance divided by the sum.
contribution_rows = []
for (category, model), grp in comparison_df.groupby(["Category", "Model"]):
    # Use only complete six-property model/category groups for directly comparable shares.
    prop_map = grp.set_index("Property")["Wasserstein_norm"].reindex(PROPERTIES)
    if prop_map.notna().sum() != len(PROPERTIES):
        continue
    total_distance = prop_map.sum()
    for prop, value in prop_map.items():
        share = (100.0 * value / total_distance) if total_distance > 0 else np.nan
        contribution_rows.append({
            "Category": category,
            "Model": model,
            "Property": prop,
            "Wasserstein_norm": value,
            "Contribution_to_Distance_%": share,
        })

feature_contribution_df = pd.DataFrame(contribution_rows)
if not feature_contribution_df.empty:
    feature_contribution_df["Contribution_to_Distance_%"] = feature_contribution_df["Contribution_to_Distance_%"].round(2)

print("Feature contribution to normalized-Wasserstein composite distance:")
display(feature_contribution_df.sort_values(["Category", "Model", "Contribution_to_Distance_%"], ascending=[True, True, False]))

# Mean contribution across evaluable models (descriptive only; model/category sample sizes differ)
mean_feature_contribution = (
    feature_contribution_df.groupby("Property", as_index=False)["Contribution_to_Distance_%"]
    .mean()
    .sort_values("Contribution_to_Distance_%", ascending=False)
)
print("Mean effective contribution across evaluable model-category groups:")
display(mean_feature_contribution)

# ---------- B) Leave-one-feature-out sensitivity ----------
# Baseline = all six features. Then six scenarios remove one property temporarily.
# No model is retrained; only the composite score is recalculated.
complete_groups = []
for (category, model), grp in comparison_df.groupby(["Category", "Model"]):
    available = set(grp.loc[grp["Wasserstein_norm"].notna(), "Property"])
    if set(PROPERTIES).issubset(available):
        complete_groups.append((category, model))

sensitivity_rows = []
scenarios = [("All_6_features", None)] + [(f"Without_{p}", p) for p in PROPERTIES]

for category, model in complete_groups:
    grp = comparison_df[(comparison_df["Category"] == category) & (comparison_df["Model"] == model)].copy()
    for scenario, excluded_prop in scenarios:
        use = grp if excluded_prop is None else grp[grp["Property"] != excluded_prop]
        sensitivity_rows.append({
            "Category": category,
            "Model": model,
            "Scenario": scenario,
            "Excluded_Feature": "None" if excluded_prop is None else excluded_prop,
            "N_Features": len(use),
            "Composite_Distance": use["Wasserstein_norm"].mean(),
            "Composite_Compliance_%": use["Compliance_%_in_Natural_P5-P95"].mean(),
        })

feature_sensitivity_df = pd.DataFrame(sensitivity_rows)

# Rank models independently inside each category/scenario.
if not feature_sensitivity_df.empty:
    feature_sensitivity_df["Rank_by_Distance"] = (
        feature_sensitivity_df.groupby(["Category", "Scenario"])["Composite_Distance"]
        .rank(ascending=True, method="min").astype(int)
    )
    feature_sensitivity_df["Rank_by_Compliance"] = (
        feature_sensitivity_df.groupby(["Category", "Scenario"])["Composite_Compliance_%"]
        .rank(ascending=False, method="min").astype(int)
    )

    baseline = feature_sensitivity_df[feature_sensitivity_df["Scenario"] == "All_6_features"][
        ["Category", "Model", "Rank_by_Distance", "Rank_by_Compliance", "Composite_Distance", "Composite_Compliance_%"]
    ].rename(columns={
        "Rank_by_Distance": "Baseline_Rank_by_Distance",
        "Rank_by_Compliance": "Baseline_Rank_by_Compliance",
        "Composite_Distance": "Baseline_Composite_Distance",
        "Composite_Compliance_%": "Baseline_Composite_Compliance_%",
    })

    feature_sensitivity_df = feature_sensitivity_df.merge(baseline, on=["Category", "Model"], how="left")
    feature_sensitivity_df["Delta_Rank_Distance"] = feature_sensitivity_df["Rank_by_Distance"] - feature_sensitivity_df["Baseline_Rank_by_Distance"]
    feature_sensitivity_df["Delta_Rank_Compliance"] = feature_sensitivity_df["Rank_by_Compliance"] - feature_sensitivity_df["Baseline_Rank_by_Compliance"]
    feature_sensitivity_df["Delta_Composite_Distance"] = feature_sensitivity_df["Composite_Distance"] - feature_sensitivity_df["Baseline_Composite_Distance"]
    feature_sensitivity_df["Delta_Composite_Compliance_%"] = feature_sensitivity_df["Composite_Compliance_%"] - feature_sensitivity_df["Baseline_Composite_Compliance_%"]

# Spearman rank stability relative to all-six-feature baseline.
stability_rows = []
for category in feature_sensitivity_df["Category"].unique() if not feature_sensitivity_df.empty else []:
    base = feature_sensitivity_df[(feature_sensitivity_df["Category"] == category) & (feature_sensitivity_df["Scenario"] == "All_6_features")]
    for scenario in [s for s in feature_sensitivity_df["Scenario"].unique() if s != "All_6_features"]:
        cur = feature_sensitivity_df[(feature_sensitivity_df["Category"] == category) & (feature_sensitivity_df["Scenario"] == scenario)]
        merged = base[["Model", "Rank_by_Distance", "Rank_by_Compliance"]].merge(
            cur[["Model", "Rank_by_Distance", "Rank_by_Compliance"]], on="Model", suffixes=("_base", "_loo")
        )
        if len(merged) >= 2:
            rho_d, _ = stats.spearmanr(merged["Rank_by_Distance_base"], merged["Rank_by_Distance_loo"])
            rho_c, _ = stats.spearmanr(merged["Rank_by_Compliance_base"], merged["Rank_by_Compliance_loo"])
        else:
            rho_d = rho_c = np.nan
        stability_rows.append({
            "Category": category,
            "Scenario": scenario,
            "Excluded_Feature": scenario.replace("Without_", ""),
            "N_Models": len(merged),
            "Spearman_Rank_Stability_Distance": rho_d,
            "Spearman_Rank_Stability_Compliance": rho_c,
            "Max_Abs_Delta_Rank_Distance": cur["Delta_Rank_Distance"].abs().max() if len(cur) else np.nan,
            "Max_Abs_Delta_Rank_Compliance": cur["Delta_Rank_Compliance"].abs().max() if len(cur) else np.nan,
        })

sensitivity_stability_df = pd.DataFrame(stability_rows)

print("Leave-one-feature-out sensitivity (baseline + six exclusions):")
display(feature_sensitivity_df.sort_values(["Category", "Scenario", "Rank_by_Distance"]))
print("Rank-stability summary:")
display(sensitivity_stability_df)

feature_contribution_df.to_csv("feature_contribution_analysis.csv", index=False)
feature_sensitivity_df.to_csv("feature_sensitivity_analysis.csv", index=False)
sensitivity_stability_df.to_csv("feature_sensitivity_rank_stability.csv", index=False)
print("Saved: feature_contribution_analysis.csv")
print("Saved: feature_sensitivity_analysis.csv")
print("Saved: feature_sensitivity_rank_stability.csv")


Feature contribution to normalized-Wasserstein composite distance:


,Category,Model,Property,Wasserstein_norm,Contribution_to_Distance_%
2,AMP,AMP,GRAVY,2.110,29.63
4,AMP,AMP,NetCharge,1.701,23.89
3,AMP,AMP,pI,1.407,19.76
5,AMP,AMP,AliphaticIndex,0.849,11.92
0,AMP,AMP,Length,0.639,8.97
1,AMP,AMP,MW,0.414,5.81
10,AMP,AMPAINTER,NetCharge,1.791,27.76
9,AMP,AMPAINTER,pI,1.661,25.75
8,AMP,AMPAINTER,GRAVY,1.451,22.49
6,AMP,AMPAINTER,Length,0.585,9.07


Mean effective contribution across evaluable model-category groups:


,Property,Contribution_to_Distance_%
5,pI,27.36000
4,NetCharge,18.02750
1,GRAVY,16.90125
2,Length,14.37500
3,MW,12.67625
0,AliphaticIndex,10.66000


Leave-one-feature-out sensitivity (baseline + six exclusions):


,Category,Model,Scenario,Excluded_Feature,N_Features,Composite_Distance,Composite_Compliance_%,Rank_by_Distance,Rank_by_Compliance,Baseline_Rank_by_Distance,Baseline_Rank_by_Compliance,Baseline_Composite_Distance,Baseline_Composite_Compliance_%,Delta_Rank_Distance,Delta_Rank_Compliance,Delta_Composite_Distance,Delta_Composite_Compliance_%
21,AMP,HMAMP,All_6_features,None,6,0.556500,85.200000,1,3,1,3,0.556500,85.200000,0,0,0.000000,0.000000
14,AMP,DIFF,All_6_features,None,6,0.777000,75.016667,2,4,2,4,0.777000,75.016667,0,0,0.000000,0.000000
28,AMP,PEPGAN,All_6_features,None,6,0.840000,97.216667,3,1,3,1,0.840000,97.216667,0,0,0.000000,0.000000
35,AMP,RR,All_6_features,None,6,0.941167,88.900000,4,2,4,2,0.941167,88.900000,0,0,0.000000,0.000000
7,AMP,AMPAINTER,All_6_features,None,6,1.075167,74.450000,5,5,5,5,1.075167,74.450000,0,0,0.000000,0.000000
0,AMP,AMP,All_6_features,None,6,1.186667,66.683333,6,6,6,6,1.186667,66.683333,0,0,0.000000,0.000000
27,AMP,HMAMP,Without_AliphaticIndex,AliphaticIndex,5,0.588800,84.460000,1,3,1,3,0.556500,85.200000,0,0,0.032300,-0.740000
20,AMP,DIFF,Without_AliphaticIndex,AliphaticIndex,5,0.808000,73.360000,2,5,2,4,0.777000,75.016667,0,1,0.031000,-1.656667
34,AMP,PEPGAN,Without_AliphaticIndex,AliphaticIndex,5,0.845000,96.660000,3,1,3,1,0.840000,97.216667,0,0,0.005000,-0.556667
41,AMP,RR,Without_AliphaticIndex,AliphaticIndex,5,1.052000,86.680000,4,2,4,2,0.941167,88.900000,0,0,0.110833,-2.220000


Rank-stability summary:


,Category,Scenario,Excluded_Feature,N_Models,Spearman_Rank_Stability_Distance,Spearman_Rank_Stability_Compliance,Max_Abs_Delta_Rank_Distance,Max_Abs_Delta_Rank_Compliance
0,AMP,Without_Length,Length,6,1.000000,0.985611,0,1
1,AMP,Without_MW,MW,6,1.000000,0.985611,0,1
2,AMP,Without_GRAVY,GRAVY,6,1.000000,0.828571,0,2
3,AMP,Without_pI,pI,6,1.000000,0.942857,0,1
4,AMP,Without_NetCharge,NetCharge,6,0.885714,1.000000,1,0
5,AMP,Without_AliphaticIndex,AliphaticIndex,6,1.000000,0.942857,0,1
6,Cyclic,Without_Length,Length,2,1.000000,1.000000,0,0
7,Cyclic,Without_MW,MW,2,1.000000,1.000000,0,0
8,Cyclic,Without_GRAVY,GRAVY,2,1.000000,1.000000,0,0
9,Cyclic,Without_pI,pI,2,-1.000000,1.000000,1,0


Saved: feature_contribution_analysis.csv
Saved: feature_sensitivity_analysis.csv
Saved: feature_sensitivity_rank_stability.csv


In [10]:
#@title 🧭 7. Generation-Strategy Comparison & Ranking { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-stratcmp {
        background: linear-gradient(135deg, #bfe6fb, #cdeffd, #e0f7ff);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(125, 211, 252, 0.4);
        margin-bottom: 16px;
    }
    .title-stratcmp {
        text-align: center;
        color: #0284c7;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-stratcmp {
        background: white;
        color: #0284c7;
        border: 1px solid #bae6fd;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-stratcmp:hover {
        background: #e0f2fe;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-stratcmp" id="headerBox_stratcmp">
    <div class="title-stratcmp">🧭 7. Generation-Strategy Comparison &amp; Ranking (NEW)</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #0284c7, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        Same pipeline as above, but peptides are pooled by <b>Strategy</b> (RL / Pseudo-RL / Hybrid and Advanced /
        Other) instead of individual Model. Answers: which generation strategy, as a whole, best reproduces natural-like
        peptide properties — regardless of which specific model within it produced a given peptide. Kept as a separate
        table/plot from the Model-level results, not merged.
    </div>
</div>
<button class="toggle-btn-stratcmp" onclick="toggleHeader_stratcmp()">Hide card</button>
<script>
function toggleHeader_stratcmp() {
    const box = document.getElementById("headerBox_stratcmp");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


comparison_rows_strat = []
for category in sorted(df["Category"].unique()):
    nat_sub = df[(df["Category"] == category) & (df["Model"] == "Natural")]
    strategies = [s for s in df[df["Category"] == category]["Strategy"].unique() if s != "Natural"]
    for prop in PROPERTIES:
        nat_vals = nat_sub[prop].dropna().values
        if len(nat_vals) < 3:
            continue
        p5, p95 = np.percentile(nat_vals, [5, 95])
        nat_std = np.std(nat_vals) if np.std(nat_vals) > 0 else np.nan
        for strategy in strategies:
            strat_vals = df[(df["Category"] == category) & (df["Strategy"] == strategy)][prop].dropna().values
            if len(strat_vals) < 3:
                continue
            mw_u, mw_p = stats.mannwhitneyu(strat_vals, nat_vals, alternative="two-sided")
            ks_stat, ks_p = stats.ks_2samp(strat_vals, nat_vals)
            wass = stats.wasserstein_distance(strat_vals, nat_vals)
            wass_norm = wass / nat_std if nat_std and not np.isnan(nat_std) else np.nan
            compliance = float(np.mean((strat_vals >= p5) & (strat_vals <= p95)) * 100)
            comparison_rows_strat.append({
                "Category": category, "Strategy": strategy, "Property": prop,
                "N_Strategy": len(strat_vals), "N_Natural": len(nat_vals),
                "MannWhitney_p": round(mw_p, 4), "Significant_Diff (p<0.05)": mw_p < 0.05,
                "KS_stat": round(ks_stat, 3), "KS_p": round(ks_p, 4),
                "Wasserstein_norm": round(wass_norm, 3) if not np.isnan(wass_norm) else np.nan,
                "Compliance_%_in_Natural_P5-P95": round(compliance, 1),
            })
comparison_df_strategy = pd.DataFrame(comparison_rows_strat)

composite_rows_strat = []
for (category, strategy), grp in comparison_df_strategy.groupby(["Category", "Strategy"]):
    composite_rows_strat.append({
        "Category": category, "Strategy": strategy,
        "Composite_Compliance_%": round(grp["Compliance_%_in_Natural_P5-P95"].mean(), 1),
        "Composite_Distance": round(grp["Wasserstein_norm"].mean(skipna=True), 3),
        "Properties_Significantly_Different": int(grp["Significant_Diff (p<0.05)"].sum()),
        "Properties_Tested": len(grp),
    })
composite_df_strategy = pd.DataFrame(composite_rows_strat)

rank_rows_strat = []
for category, grp in composite_df_strategy.groupby("Category"):
    grp = grp.copy()
    grp["Rank_by_Compliance"] = grp["Composite_Compliance_%"].rank(ascending=False, method="min").astype(int)
    grp["Rank_by_Distance"] = grp["Composite_Distance"].rank(ascending=True, method="min").astype(int)
    rank_rows_strat.append(grp)
rank_df_strategy = pd.concat(rank_rows_strat).sort_values(["Category", "Rank_by_Compliance"])
print("Strategy ranking:"); display(rank_df_strategy)


Strategy ranking:


,Category,Strategy,Composite_Compliance_%,Composite_Distance,Properties_Significantly_Different,Properties_Tested,Rank_by_Compliance,Rank_by_Distance
1,AMP,RL,85.6,0.758,5,6,1,1
0,AMP,Hybrid and Advanced,74.5,0.904,5,6,2,2
2,Cyclic,Hybrid and Advanced,54.8,2.129,6,6,1,1


In [11]:
#@title 🎨 8. Visualizations { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-viz {
        background: linear-gradient(135deg, #ffd1dc, #ffc2d1, #ffe4ec);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(255, 182, 193, 0.4);
        margin-bottom: 16px;
    }
    .title-viz {
        text-align: center;
        color: #db2777;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-viz {
        background: white;
        color: #db2777;
        border: 1px solid #fbcfe8;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-viz:hover {
        background: #fce7f3;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-viz" id="headerBox_viz">
    <div class="title-viz">🎨 8. Visualizations</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #db2777, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        One cell defines and immediately renders all five plots: (1) KDE distribution overlay, (2) empirical CDF,
        (3) Mann-Whitney significance heatmap, (4) Model ranking bars, (5) Strategy ranking bars (NEW). Function
        definitions and their execution are merged since they are only ever used once, right here.
    </div>
</div>
<button class="toggle-btn-viz" onclick="toggleHeader_viz()">Hide card</button>
<script>
function toggleHeader_viz() {
    const box = document.getElementById("headerBox_viz");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode("utf-8")
    plt.close(fig)
    return f'<img src="data:image/png;base64,{encoded}" style="max-width:100%;margin:10px 0;">'

def model_order(models):
    return sorted(models, key=lambda m: (m != "Natural", m))

def strategy_order(strategies):
    priority = {"RL": 0, "Pseudo-RL": 1, "Hybrid and Advanced": 2, "Other": 3}
    return sorted(strategies, key=lambda s: priority.get(s, 99))

def plot_distributions(df):
    categories = sorted(df["Category"].unique())
    n_cats, n_props = len(categories), len(PROPERTIES)
    fig, axes = plt.subplots(n_props, n_cats, figsize=(4.5 * n_cats, 3.2 * n_props), squeeze=False)
    fig.suptitle("Distribution Comparison: Natural Reference vs Generated Models", fontsize=14, fontweight="bold", y=1.01)
    for col_i, cat in enumerate(categories):
        sub = df[df["Category"] == cat]
        models = model_order(sub["Model"].unique())
        non_natural = [m for m in models if m != "Natural"]
        colors = plt.cm.tab10(np.linspace(0, 0.9, max(len(non_natural), 1)))
        color_map = {m: colors[i] for i, m in enumerate(non_natural)}
        for row_i, prop in enumerate(PROPERTIES):
            ax = axes[row_i][col_i]
            for m in models:
                vals = sub[sub["Model"] == m][prop].dropna().values
                if len(vals) < 2 or np.std(vals) == 0:
                    continue
                try:
                    if m == "Natural":
                        sns.kdeplot(vals, ax=ax, fill=True, color="#7f8c8d", alpha=0.3, linewidth=2, label="Natural")
                    else:
                        sns.kdeplot(vals, ax=ax, fill=False, color=color_map[m], linewidth=1.8, label=m)
                except Exception:
                    continue
            ax.set_title(f"{cat} - {prop}", fontsize=9, fontweight="bold")
            ax.set_xlabel(prop, fontsize=8); ax.set_ylabel("Density", fontsize=8)
            if row_i == 0 and col_i == n_cats - 1:
                ax.legend(fontsize=7, loc="upper right")
            ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig

def plot_ecdf(df):
    categories = sorted(df["Category"].unique())
    n_cats, n_props = len(categories), len(PROPERTIES)
    fig, axes = plt.subplots(n_props, n_cats, figsize=(4.5 * n_cats, 3.2 * n_props), squeeze=False)
    fig.suptitle("Empirical CDF: Natural Reference vs Generated Models", fontsize=14, fontweight="bold", y=1.01)
    for col_i, cat in enumerate(categories):
        sub = df[df["Category"] == cat]
        models = model_order(sub["Model"].unique())
        non_natural = [m for m in models if m != "Natural"]
        colors = plt.cm.tab10(np.linspace(0, 0.9, max(len(non_natural), 1)))
        color_map = {m: colors[i] for i, m in enumerate(non_natural)}
        for row_i, prop in enumerate(PROPERTIES):
            ax = axes[row_i][col_i]
            for m in models:
                vals = np.sort(sub[sub["Model"] == m][prop].dropna().values)
                if len(vals) < 2:
                    continue
                y = np.arange(1, len(vals) + 1) / len(vals)
                if m == "Natural":
                    ax.step(vals, y, where="post", color="#2c3e50", linewidth=2.5, linestyle="--", label="Natural")
                else:
                    ax.step(vals, y, where="post", color=color_map[m], linewidth=1.6, label=m)
            ax.set_title(f"{cat} - {prop}", fontsize=9, fontweight="bold")
            ax.set_xlabel(prop, fontsize=8); ax.set_ylabel("Cumulative Probability", fontsize=8)
            if row_i == 0 and col_i == n_cats - 1:
                ax.legend(fontsize=7, loc="lower right")
            ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig

def plot_significance_heatmap(comparison_df):
    categories = sorted(comparison_df["Category"].unique())
    # ارتفاع فیگور را بر اساس بیشترین تعداد مدل در هر Category، به‌صورت پویا تنظیم می‌کنیم
    # تا وقتی تعداد مدل‌ها (ردیف‌های heatmap) زیاد است، لیبل‌ها روی هم نیفتند.
    n_models_max = comparison_df.groupby("Category")["Model"].nunique().max()
    fig_height = max(4.5, 0.75 * n_models_max + 1.5)
    fig, axes = plt.subplots(1, len(categories), figsize=(7 * len(categories), fig_height), squeeze=False)
    fig.suptitle("Mann-Whitney U Test: p-values vs Natural Distribution (per Model)", fontsize=13, fontweight="bold")
    for i, cat in enumerate(categories):
        ax = axes[0][i]
        sub = comparison_df[comparison_df["Category"] == cat]
        pivot = sub.pivot_table(index="Model", columns="Property", values="MannWhitney_p").reindex(columns=PROPERTIES)
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=0.2, linewidths=0.5, ax=ax,
                    cbar_kws={"label": "p-value"})
        ax.axhline(0, color="black", linewidth=0.5)
        ax.set_title(f"{cat}\n(green = p>0.05, no significant difference from Natural)", fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("")
        # --- رفع مشکل هم‌پوشانی لیبل‌های محور Y (اسم مدل‌ها) ---
        ax.tick_params(axis='y', rotation=0, labelsize=10)
        ax.tick_params(axis='x', rotation=45, labelsize=10)
        for label in ax.get_xticklabels():
            label.set_ha("right")
    plt.tight_layout()
    return fig

def plot_ranking(rl_rank_df):
    categories = sorted(rl_rank_df["Category"].unique())
    fig, axes = plt.subplots(1, len(categories), figsize=(5.5 * len(categories), 5), squeeze=False)
    fig.suptitle("Model Ranking: Similarity to Natural Peptide Distribution", fontsize=13, fontweight="bold")
    for i, cat in enumerate(categories):
        ax = axes[0][i]
        sub = rl_rank_df[rl_rank_df["Category"] == cat].sort_values("Composite_Compliance_%", ascending=True)
        bars = ax.barh(sub["Model"], sub["Composite_Compliance_%"], color="#2980b9", alpha=0.85)
        for bar, v in zip(bars, sub["Composite_Compliance_%"]):
            ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, f"{v:.1f}%", va="center", fontsize=8)
        ax.set_xlim(0, 110)
        ax.axvline(100, color="#7f8c8d", linestyle="--", linewidth=2, alpha=0.8)
        ax.text(100, len(sub) - 0.3, "Natural reference (100%)", color="#555555", fontsize=7, ha="right", va="bottom")
        ax.set_xlabel("% of values inside Natural P5-P95 range (avg across properties)", fontsize=8)
        ax.set_title(cat, fontsize=11, fontweight="bold")
        ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    return fig

def plot_strategy_ranking(rank_df_strategy):
    categories = sorted(rank_df_strategy["Category"].unique())
    fig, axes = plt.subplots(1, len(categories), figsize=(5.5 * len(categories), 4.5), squeeze=False)
    fig.suptitle("Generation Strategy Ranking: Similarity to Natural Peptide Distribution", fontsize=13, fontweight="bold")
    strat_colors = {"RL": "#2980b9", "Pseudo-RL": "#8e44ad", "Hybrid and Advanced": "#16a085", "Other": "#95a5a6"}
    for i, cat in enumerate(categories):
        ax = axes[0][i]
        sub = rank_df_strategy[rank_df_strategy["Category"] == cat].sort_values("Composite_Compliance_%", ascending=True)
        colors = [strat_colors.get(s, "#7f8c8d") for s in sub["Strategy"]]
        bars = ax.barh(sub["Strategy"], sub["Composite_Compliance_%"], color=colors, alpha=0.9)
        for bar, v in zip(bars, sub["Composite_Compliance_%"]):
            ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, f"{v:.1f}%", va="center", fontsize=9)
        ax.set_xlim(0, 110)
        ax.axvline(100, color="#7f8c8d", linestyle="--", linewidth=2, alpha=0.8)
        ax.set_xlabel("% of values inside Natural P5-P95 range (avg across properties)", fontsize=8)
        ax.set_title(cat, fontsize=11, fontweight="bold")
        ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    return fig

print("Generating plots...")
img_dist     = fig_to_base64(plot_distributions(df))
img_ecdf     = fig_to_base64(plot_ecdf(df))
img_sig      = fig_to_base64(plot_significance_heatmap(comparison_df))
img_rank     = fig_to_base64(plot_ranking(rl_rank_df))
img_strategy = fig_to_base64(plot_strategy_ranking(rank_df_strategy))
print("Plots done")


Generating plots...
Plots done


In [12]:
#@title 📑 9. Build & Download HTML Report { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-report {
        background: linear-gradient(135deg, #bfe6fb, #cdeffd, #e0f7ff);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(125, 211, 252, 0.4);
        margin-bottom: 16px;
    }
    .title-report {
        text-align: center;
        color: #0284c7;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-report {
        background: white;
        color: #0284c7;
        border: 1px solid #bae6fd;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-report:hover {
        background: #e0f2fe;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-report" id="headerBox_report">
    <div class="title-report">📑 9. Build &amp; Download HTML Report</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #0284c7, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        Builds a publication-ready HTML report using only the corrected sequence-based estimated pI in the primary analysis. The full before/after pI audit is exported separately as Supplementary material.
    </div>
</div>
<button class="toggle-btn-report" onclick="toggleHeader_report()">Hide card</button>
<script>
function toggleHeader_report() {
    const box = document.getElementById("headerBox_report");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


for _var in ("img_dist", "img_ecdf", "img_sig", "img_rank", "img_strategy"):
    if _var not in globals():
        raise RuntimeError(
            "Run the '8. Visualizations' cell above first (it must finish and print "
            "'Plots done') before running this report cell."
        )

html = """
<html>
<head>
<meta charset="UTF-8">
<style>
body { font-family: Arial, sans-serif; background:linear-gradient(135deg,#fdf2f8 0%,#e0f2fe 100%); margin:20px; color:#334155; }
h1 { color:#db2777; border-bottom:3px solid #0ea5e9; padding-bottom:10px; }
h2 { color:#0369a1; border-bottom:2px solid #fbcfe8; padding-bottom:5px; margin-top:40px; }
h3 { color:#be185d; margin-top:25px; }
p.note { font-size:13px; color:#64748b; }
table {
  border-collapse:collapse; width:100%; margin-bottom:30px;
  background:white; box-shadow:0 2px 8px rgba(219,39,119,0.12); font-size:13px;
}
th { background-color:#0284c7; color:white; padding:10px 8px; white-space:nowrap; }
td { padding:8px; text-align:center; }
tr:nth-child(even) { background:#fdf2f8; }
tr:hover { background:#fce7f3; }
</style>
</head>
<body>
<h1>Property-Based Benchmarking: Generated Peptides vs. Natural Peptides</h1>
<p class="note">Two independent benchmarks are reported: (1) compliance with fixed literature-defined ranges,
and (2) statistical similarity to the uploaded Natural reference set for each category (AMP/MHC/Cyclic).
Both are shown at the Model level and, separately, at the Generation-Strategy level.</p>
"""

html += "<h2>Literature Compliance Table</h2>"
html += "<p class='note'>Fixed, absolute thresholds from published literature.</p>"
html += literature_table.to_html(index=False)

html += "<h2>Natural Reference Distribution (per Category x Property)</h2>"
html += natural_ref_df.to_html(index=False)

html += "<h2>Model-Level Analysis</h2>"
html += "<h3>Plot 1 - Distribution Overlay (KDE)</h3>" + img_dist
html += "<h3>Plot 2 - Empirical CDF (ECDF)</h3>" + img_ecdf
html += "<h3>Plot 3 - Statistical Significance (Mann-Whitney U vs Natural)</h3>" + img_sig
html += "<h3>Plot 4 - Composite Ranking (Similarity to Natural)</h3>" + img_rank
html += "<h3>Full Ranking Table</h3>"
html += rl_rank_df[["Category", "Model", "Composite_Compliance_%", "Composite_Distance",
                     "Properties_Significantly_Different", "Properties_Tested",
                     "Rank_by_Compliance", "Rank_by_Distance"]].to_html(index=False)
html += "<h3>Best Model Per Category</h3>" + best_per_category.to_html(index=False)
html += "<h3>Full Statistical Comparison Table (Model vs Natural, per Property)</h3>" + comparison_df.to_html(index=False)


html += "<h2>pI Quality Control</h2>"
html += """
<p class='note'>
The primary benchmark uses only the corrected sequence-based estimated pI. Original
ProteinAnalysis.isoelectric_point() values are retained only in the separate Supplementary
audit to document the ~4.05 lower-bound artifact. The compact table below summarizes
quality control for the Natural Cyclic reference set.
</p>
"""
html += pi_qc_summary.to_html(index=False)

html += "<h2>Feature Contribution and Sensitivity Analysis</h2>"
html += "<p class='note'>No feature-importance coefficients are introduced. Percentage contribution is derived from the six equal-weight normalized Wasserstein components. Leave-one-feature-out sensitivity temporarily omits one feature and recomputes the score as the arithmetic mean of the remaining five; generative models are not retrained.</p>"
html += "<h3>Feature Contribution to Composite Distance</h3>" + feature_contribution_df.to_html(index=False)
html += "<h3>Leave-One-Feature-Out Results</h3>" + feature_sensitivity_df.to_html(index=False)
html += "<h3>Rank Stability Summary</h3>" + sensitivity_stability_df.to_html(index=False)

html += "<h2>Generation-Strategy-Level Analysis (RL vs Pseudo-RL vs Hybrid and Advanced)</h2>"
html += "<h3>Plot 5 - Strategy Ranking (Similarity to Natural)</h3>" + img_strategy
html += "<h3>Strategy Ranking Table</h3>" + rank_df_strategy.to_html(index=False)
html += "<h3>Full Statistical Comparison Table (Strategy vs Natural, per Property)</h3>" + comparison_df_strategy.to_html(index=False)

html += f"<h2>All Peptides ({len(df)} sequences)</h2>"
html += df.to_html(index=False)

html += """
<h2>Method Notes</h2>
<ul>
  <li><b>Literature-defined ranges:</b> fixed absolute thresholds from published studies (Literature Compliance Table).</li>
  <li><b>Mann-Whitney U test:</b> non-parametric test for whether values are drawn from the same distribution
  as the natural peptides.</li>
  <li><b>Kolmogorov-Smirnov test:</b> tests for differences in overall distribution shape/location.</li>
  <li><b>Sequence-based estimated pI:</b> calculated with Biopython's charge model and bisection over pH 0-14, avoiding the default ~4.05 lower-bound artifact for acidic peptides.</li>
  <li><b>Cyclic-peptide limitation:</b> pI and net-charge estimates retain Biopython's ionization model and therefore assume free ionizable N- and C-termini. For head-to-tail cyclized peptides this may not represent the physical topology; values are therefore interpreted as sequence-based estimates unless topology-specific terminal chemistry is modeled.</li>
  <li><b>Wasserstein distance (normalized by natural SD):</b> continuous measure of distance; 0 = identical.</li>
  <li><b>Compliance %:</b> percentage of values falling inside the 5th-95th percentile range of the natural peptides.</li>
  <li><b>Model vs. Strategy:</b> Category is biological; Strategy is a generation-method dimension, tracked
  independently since one strategy can produce peptides across multiple categories.</li>
</ul>
</body></html>
"""

output = "peptide_benchmarking_report_PUBLICATION_READY.html"
with open(output, "w", encoding="utf-8") as f:
    f.write(html)

files.download(output)
print(f"\nPublication-ready report saved: {output}")
print("Full pI audit: Supplementary_pI_audit_before_after.csv")
print("\n--- Best Model Per Category ---")
print(best_per_category.to_string(index=False))
print("\n--- Strategy Ranking Per Category ---")
print(rank_df_strategy[["Category","Strategy","Composite_Compliance_%","Rank_by_Compliance"]].to_string(index=False))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Publication-ready report saved: peptide_benchmarking_report_PUBLICATION_READY.html
Full pI audit: Supplementary_pI_audit_before_after.csv

--- Best Model Per Category ---
Category  Model  Composite_Compliance_%  Composite_Distance
     AMP PEPGAN                    97.2               0.840
  Cyclic    CYC                    63.0               1.633

--- Strategy Ranking Per Category ---
Category            Strategy  Composite_Compliance_%  Rank_by_Compliance
     AMP                  RL                    85.6                   1
     AMP Hybrid and Advanced                    74.5                   2
  Cyclic Hybrid and Advanced                    54.8                   1


In [13]:
# ============================================================
# Supplementary pI Audit CSV
# Creates the raw before-vs-after pI audit for publication
# ============================================================

import numpy as np
import pandas as pd
from Bio.SeqUtils.ProtParam import ProteinAnalysis

supp_rows = []

for _, row in df.iterrows():
    seq = row["CleanSeq"]
    analysis = ProteinAnalysis(seq)

    # Original/default Biopython pI
    pi_original = analysis.isoelectric_point()

    # Corrected pI already used in the final benchmark
    pi_corrected = row["pI"]

    supp_rows.append({
        "ID": row["ID"],
        "Category": row["Category"],
        "Model": row["Model"],
        "Strategy": row["Strategy"] if "Strategy" in row.index else np.nan,
        "Sequence": seq,
        "pI_original_Biopython": pi_original,
        "pI_corrected": pi_corrected,
        "Difference_corrected_minus_original": pi_corrected - pi_original,
        "Charge_at_pH_4.05": analysis.charge_at_pH(4.05),
        "Original_at_4.050028": bool(
            np.isclose(pi_original, 4.050028, atol=1e-6)
        ),
        "Corrected_at_4.050028": bool(
            np.isclose(pi_corrected, 4.050028, atol=1e-6)
        ),
    })

pI_audit_before_after = pd.DataFrame(supp_rows)

# Save publication supplementary file
supplementary_filename = "pI_audit_before_after.csv"
pI_audit_before_after.to_csv(
    supplementary_filename,
    index=False
)

print("=" * 80)
print("SUPPLEMENTARY pI AUDIT CREATED")
print("=" * 80)

print(f"Saved file: {supplementary_filename}")
print(f"Number of peptide sequences: {len(pI_audit_before_after)}")

print(
    "Original pI ≈ 4.050028:",
    int(pI_audit_before_after["Original_at_4.050028"].sum())
)

print(
    "Corrected pI ≈ 4.050028:",
    int(pI_audit_before_after["Corrected_at_4.050028"].sum())
)

# Specific QC for Natural Cyclic
cyclic_qc = pI_audit_before_after[
    (pI_audit_before_after["Category"] == "Cyclic") &
    (pI_audit_before_after["Model"] == "Natural")
].copy()

print("\nNatural Cyclic QC:")
print(
    "Original pI ≈ 4.050028:",
    int(cyclic_qc["Original_at_4.050028"].sum()),
    "/",
    len(cyclic_qc)
)

print(
    "Corrected pI ≈ 4.050028:",
    int(cyclic_qc["Corrected_at_4.050028"].sum()),
    "/",
    len(cyclic_qc)
)

print(
    "Corrected mean ± SD:",
    f"{cyclic_qc['pI_corrected'].mean():.3f} ± "
    f"{cyclic_qc['pI_corrected'].std(ddof=1):.3f}"
)

display(pI_audit_before_after.head(20))

SUPPLEMENTARY pI AUDIT CREATED
Saved file: pI_audit_before_after.csv
Number of peptide sequences: 443
Original pI ≈ 4.050028: 106
Corrected pI ≈ 4.050028: 0

Natural Cyclic QC:
Original pI ≈ 4.050028: 94 / 100
Corrected pI ≈ 4.050028: 0 / 100
Corrected mean ± SD: 3.606 ± 0.277


,ID,Category,Model,Strategy,Sequence,pI_original_Biopython,pI_corrected,Difference_corrected_minus_original,Charge_at_pH_4.05,Original_at_4.050028,Corrected_at_4.050028
0,HMAMP_1,AMP,HMAMP,RL,RKVFRTVVP,11.999968,12.008463,0.008495,3.239897,False,False
1,HMAMP_2,AMP,HMAMP,RL,IRAILPLARKWNTVWS,11.999968,12.008463,0.008495,3.239897,False,False
2,HMAMP_3,AMP,HMAMP,RL,ILKYCKRTV,9.787026,9.787036,0.000010,3.239884,False,False
3,HMAMP_5,AMP,HMAMP,RL,CKDWAWKNYYKKYKG,9.583113,9.583139,0.000027,4.739878,False,False
4,HMAMP_6,AMP,HMAMP,RL,KWFVVWISKLVSKLSNNP,10.302064,10.302056,-0.000008,3.239895,False,False
5,HMAMP_7,AMP,HMAMP,RL,LLPLLLKFFLSKSTV,10.002737,10.002721,-0.000016,2.239896,False,False
6,HMAMP_8,AMP,HMAMP,RL,ILLPALGLLPSISCSNNKTCNLLGL,8.058371,8.058389,0.000018,1.239875,False,False
7,HMAMP_9,AMP,HMAMP,RL,VLWLLFGLTWLLTKKMSEKFRRLY,10.446731,10.446708,-0.000023,4.955094,False,False
8,HMAMP_10,AMP,HMAMP,RL,KIFMQILTKIKKAAKNVSETIQTNKY,10.125550,10.125538,-0.000011,5.955143,False,False
9,Diff-AMP_1,AMP,DIFF,Hybrid and Advanced,SASLVSLTKLISPKLKS,10.301290,10.301307,0.000016,3.238933,False,False


In [14]:
from google.colab import files
files.download("pI_audit_before_after.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>